# 🧪 Datenaugmentierung verstehen – die Bilder-Werkstatt

**Worum geht es?** Wir bauen eine kleine Gradio-App, mit der du das **Prinzip der Datenaugmentierung**
hands-on erlebst. Du lädst ein paar Graustufenbilder, schiebst an Reglern, und erzeugst daraus einen
viel größeren, augmentierten Datensatz – fertig im CSV-Format für deine spätere CNN-App.

---

### Der rote Faden – drei Ideen, die hier zusammenkommen

**1️⃣ Ein Datenpunkt *ist* ein Bild.**
Deine spätere CNN-App liest CSV-Zeilen der Form `feature1, feature2, …, featureN, class`.
Wenn die Anzahl der Features eine **Quadratzahl** ist (z. B. 576 = 24×24), kann man jede Zeile
als **quadratisches Graustufenbild** auffassen. Genau diesen *Repräsentationswechsel* nutzen wir:
Bild → augmentieren → zurück in eine CSV-Zeile.

**2️⃣ Augmentierung = mehr Daten ohne neue Daten.**
Aus *einem* Bild machen wir durch **label-erhaltende** Transformationen (drehen, spiegeln, beschneiden …)
viele leicht verschiedene Varianten. Das Label bleibt dasselbe – ein gedrehter Kreis ist ein Kreis.
So lernt ein Modell **Invarianzen** und überanpasst weniger.

**3️⃣ Transformationen sind *nicht* vertauschbar.**
Ob du **erst drehst und dann eine Perspektive** rechnest oder umgekehrt, macht einen sichtbaren
Unterschied. Diese Nicht-Kommutativität machen wir am Ende zu einem kleinen Experiment.

---

### Was die App können wird
- 📂 Graustufen-PNGs aus einem Ordner laden (Dateiname = Klasse)
- 🔢 Klassennamen **label-encodieren** (Text → Integer) und in einer **JSON** sichern
- 🎚️ 7 Augmentierungen über Regler/Checkboxen steuern – mit **Live-Vorschau**
- 🔀 die **Reihenfolge** der Transformationen festlegen
- 💾 Datensatz als **CSV** speichern (Auflösung bleibt erhalten, Werte 0–255, Integer-Labels)
- 🖼️ optional die augmentierten Bilder als Graustufen-PNG mitspeichern
- ♻️ die JSON-Konfiguration später **wieder laden** und den Zustand herstellen


## 0 · Setup

Wir brauchen `torch`, `torchvision`, `gradio`, `pillow`, `numpy`, `pandas`, `matplotlib`.
Falls etwas fehlt, die nächste Zelle einmal ausführen (danach Kernel ggf. neu starten).


In [ ]:
# Bei Bedarf einkommentieren:
# %pip install torch torchvision gradio pillow numpy pandas matplotlib

import os, io, json, base64, math
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torchvision import transforms

import matplotlib
matplotlib.use("Agg")          # backend ohne Fenster – wichtig fuer Gradio im Notebook
import matplotlib.pyplot as plt

import gradio as gr

print("torch      :", torch.__version__)
print("gradio     :", gr.__version__)
print("Alles bereit ✔")

## 1 · Beispielbilder anlegen

Damit das Notebook sofort läuft, sind vier kleine **24×24-Graustufenformen** direkt eingebettet
(Base64). Die nächste Zelle schreibt sie in den Ordner `beispiel_formen/`. Der **Dateiname ist die Klasse**:

`kreis.png`, `linie.png`, `dreieck.png`, `rechteck.png`

> 👉 Eigene Daten? Lege einfach weitere Graustufen-PNGs (max. 28×28) in einen Ordner – der Dateiname
> (ohne `.png`) wird zur Klasse. Du kannst sie z. B. in Photoshop erstellen.


In [ ]:
# Eingebettete Beispielbilder (Base64-PNG) -> Ordner schreiben
_EMBEDDED = {
"kreis": "iVBORw0KGgoAAAANSUhEUgAAABgAAAAYCAAAAADFHGIkAAABCGlDQ1BJQ0MgUHJvZmlsZQAAeJxjYGA8wQAELAYMDLl5JUVB7k4KEZFRCuwPGBiBEAwSk4sLGHADoKpv1yBqL+viUYcLcKakFicD6Q9ArFIEtBxopAiQLZIOYWuA2EkQtg2IXV5SUAJkB4DYRSFBzkB2CpCtkY7ETkJiJxcUgdT3ANk2uTmlyQh3M/Ck5oUGA2kOIJZhKGYIYnBncAL5H6IkfxEDg8VXBgbmCQixpJkMDNtbGRgkbiHEVBYwMPC3MDBsO48QQ4RJQWJRIliIBYiZ0tIYGD4tZ2DgjWRgEL7AwMAVDQsIHG5TALvNnSEfCNMZchhSgSKeDHkMyQx6QJYRgwGDIYMZAKbWPz9HbOBQAAABJElEQVR4nFXMSY4cMQxE0U9KpHKowei+/yU9AJUppcILo9r220bgE+4QQEAaRmIbBlALLaJZreSdHSBInhQHB2gr1MLaqgPA04jSFvD657ETTvGEhY3Hd6mLBUs2wFh2uEvzuiRpWMGCAiuJzX5J+jkkHewVgopTp6Q51aVzKthIwCuo60sXNKMWG6gbb1eX7KD4NSzGqHoPvkIzmwSOdI6v1CERVHiALv3jEDzwcpLnHPyVvXAG0PZvh+b7/9J5DwcW8PJ/y2Ohucjt4uIFkwl0br1zQDU+GFPX/CXpegnuLIVaCRrSlI4hqbIQBk7CDdCQfnwCBdYaJFTnIwvh+HO3pHwCFGMlIGFLw5IA3hlYuDVqyySMbEAFzIyCNawRrA0KvwF9wa2+WSFI4gAAAABJRU5ErkJggg==",
"linie": "iVBORw0KGgoAAAANSUhEUgAAABgAAAAYCAAAAADFHGIkAAABCGlDQ1BJQ0MgUHJvZmlsZQAAeJxjYGA8wQAELAYMDLl5JUVB7k4KEZFRCuwPGBiBEAwSk4sLGHADoKpv1yBqL+viUYcLcKakFicD6Q9ArFIEtBxopAiQLZIOYWuA2EkQtg2IXV5SUAJkB4DYRSFBzkB2CpCtkY7ETkJiJxcUgdT3ANk2uTmlyQh3M/Ck5oUGA2kOIJZhKGYIYnBncAL5H6IkfxEDg8VXBgbmCQixpJkMDNtbGRgkbiHEVBYwMPC3MDBsO48QQ4RJQWJRIliIBYiZ0tIYGD4tZ2DgjWRgEL7AwMAVDQsIHG5TALvNnSEfCNMZchhSgSKeDHkMyQx6QJYRgwGDIYMZAKbWPz9HbOBQAAAA+0lEQVR4nH3SQW/CMAyG4Td2mrSi8P9vDFCFgAHT/tvECuq+HRhNTvPRlq08djAWEEi0RoCGORK7LmdWQEpW8nEtbTFYALRVZfjS9zF0GXqMUArhU9IAPSFThcFGD217Ih2hjIos+bhLF7wl4XUTcS+NV8Ooh2UMu47S4PjzYQEanETrXHTXwZ1m9ligI9LvNY1DMq89OdDDWdIGZk/A6CHDRtK5eKwFWOCWhnHSvngsJWDV4H7QvXgaCM82x4d/PW8ZiHP+5XlIO4vVkl+eQdPPCdp6ZTnQ4+dp0nuqzjJ7Thpvx7L+lyc0HKX6U/x5InBYL0t+9nRAR/oFm8dYf8aZTPEAAAAASUVORK5CYII=",
"dreieck": "iVBORw0KGgoAAAANSUhEUgAAABgAAAAYCAAAAADFHGIkAAABCGlDQ1BJQ0MgUHJvZmlsZQAAeJxjYGA8wQAELAYMDLl5JUVB7k4KEZFRCuwPGBiBEAwSk4sLGHADoKpv1yBqL+viUYcLcKakFicD6Q9ArFIEtBxopAiQLZIOYWuA2EkQtg2IXV5SUAJkB4DYRSFBzkB2CpCtkY7ETkJiJxcUgdT3ANk2uTmlyQh3M/Ck5oUGA2kOIJZhKGYIYnBncAL5H6IkfxEDg8VXBgbmCQixpJkMDNtbGRgkbiHEVBYwMPC3MDBsO48QQ4RJQWJRIliIBYiZ0tIYGD4tZ2DgjWRgEL7AwMAVDQsIHG5TALvNnSEfCNMZchhSgSKeDHkMyQx6QJYRgwGDIYMZAKbWPz9HbOBQAAABF0lEQVR4nG2RSUoEURBEX2b+oarbYSne/zieQMQBUVFERNESq+r/cCModMfyBURmEPCrwJzCwK48w2YPT0Yc7eFRjQoHeywu5yF2qQ1I5/huFLfrvWzfkSat1/+eIQMF1KVVJCgwGBzDBuz+TdMkNdKYGAEHM4LH91Va1n712xZ3AuPmU1KTWjfYkoGoDvVVH5L0JS0jOEEBKs9aNXc1LZqvvBQgRQbXolmSJulFkEkQMHZJi9qipq6vO8h4aqQzo5Hw6GoNpgeawZaiO83L2iVJamoXmWT1mwttpmoRyU1oOeGJU0H+m82AYg4ULK3eiVa6hDC6KYXmQ9wZyr+pRwK2IzhgJNzMzD1wKFDBw7wSgBkQmGWIH6nHkNIScw6hAAAAAElFTkSuQmCC",
"rechteck": "iVBORw0KGgoAAAANSUhEUgAAABgAAAAYCAAAAADFHGIkAAABCGlDQ1BJQ0MgUHJvZmlsZQAAeJxjYGA8wQAELAYMDLl5JUVB7k4KEZFRCuwPGBiBEAwSk4sLGHADoKpv1yBqL+viUYcLcKakFicD6Q9ArFIEtBxopAiQLZIOYWuA2EkQtg2IXV5SUAJkB4DYRSFBzkB2CpCtkY7ETkJiJxcUgdT3ANk2uTmlyQh3M/Ck5oUGA2kOIJZhKGYIYnBncAL5H6IkfxEDg8VXBgbmCQixpJkMDNtbGRgkbiHEVBYwMPC3MDBsO48QQ4RJQWJRIliIBYiZ0tIYGD4tZ2DgjWRgEL7AwMAVDQsIHG5TALvNnSEfCNMZchhSgSKeDHkMyQx6QJYRgwGDIYMZAKbWPz9HbOBQAAAAiUlEQVR4nM3SuxLCMAxE0eNHJoEw5P8/lCYkNk0AUzg1t1yNRtKuyK4kHxJXGUKyhBQOUlikAEaChsAIosGc40GeDSIu3Extx+TGhXtiqw0b6c7MqD6++qMamSFb61YOtrrKjn3iMzYzylDQKj/8byGX3f4+fLeXzJklPRP7tveCOom29wyd93kB/6tYwFTw8coAAAAASUVORK5CYII="
}

BEISPIEL_ORDNER = "beispiel_formen"
os.makedirs(BEISPIEL_ORDNER, exist_ok=True)
for _cls, _b64 in _EMBEDDED.items():
    _im = Image.open(io.BytesIO(base64.b64decode(_b64))).convert("L")
    _im.save(os.path.join(BEISPIEL_ORDNER, f"{_cls}.png"))
print("Geschrieben:", sorted(os.listdir(BEISPIEL_ORDNER)))
print("Größe Beispiel:", Image.open(os.path.join(BEISPIEL_ORDNER, "kreis.png")).size)

## 2 · Laden & Label-Encoding

**Was passiert hier?**
- Alle PNGs im Ordner werden als **Graustufe** (`mode="L"`, ein Kanal, Werte 0–255) geladen.
- Der Dateiname ohne Endung ist die **Klasse**.
- Die Klassen werden **sortiert** und auf Integer abgebildet (`label_encoding`), z. B.
  `{"dreieck": 0, "kreis": 1, "linie": 2, "rechteck": 3}`. Das brauchen wir, weil die CNN-App
  nur **Integer-Labels** verarbeitet.
- Alle Bilder müssen **gleich groß und quadratisch** sein (Feature-Zahl = Kantenlänge²).


In [ ]:
def lade_ordner(ordner):
    """Liest alle *.png als Graustufe. Rueckgabe: (items, encoding, (H, W)).
    items = Liste von (klassenname, PIL.Image 'L'). """
    if not os.path.isdir(ordner):
        raise FileNotFoundError(f"Ordner nicht gefunden: {ordner}")
    items = []
    for fn in sorted(os.listdir(ordner)):
        if fn.lower().endswith(".png"):
            klasse = os.path.splitext(fn)[0]
            bild = Image.open(os.path.join(ordner, fn)).convert("L")
            items.append((klasse, bild))
    if not items:
        raise ValueError("Keine PNG-Dateien im Ordner gefunden.")

    # Pruefung: gleich gross + quadratisch
    groessen = {im.size for _, im in items}
    if len(groessen) != 1:
        raise ValueError(f"Bilder haben unterschiedliche Größen: {groessen}")
    (W, H) = items[0][1].size            # PIL.size = (Breite, Hoehe)
    if H != W:
        raise ValueError(f"Bilder sind nicht quadratisch: {W}x{H}")
    if not math.isqrt(H * W) ** 2 == H * W:
        raise ValueError("Feature-Zahl ist keine Quadratzahl.")

    klassen = sorted({k for k, _ in items})
    encoding = {k: i for i, k in enumerate(klassen)}
    return items, encoding, (H, W)


# kurzer Test
_items, _enc, (_H, _W) = lade_ordner(BEISPIEL_ORDNER)
print("Klassen + Encoding:", _enc)
print(f"Bildgröße: {_H}x{_W}  ->  {_H*_W} Features pro Bild")
print("Anzahl Originalbilder:", len(_items))

## 3 · Die sieben Augmentierungen

Wir nutzen **`torchvision.transforms`** als Engine. Warum torchvision und nicht „von Hand"?
Weil das Lernziel das *Prinzip* ist, nicht das Implementieren von Interpolation. torchvision macht
den **Bild-Charakter** explizit, und jede Transformation ist eine kleine, benennbare Operation.

Wichtig: Manche Regler steuern eine **Stärke**, andere eine **Wahrscheinlichkeit** – steht jeweils dabei.

| Transformation | Regler steuert | Wirkung |
|---|---|---|
| `RandomResizedCrop` | **Stärke** (kleinste Crop-Skala) | schneidet zufälligen Bereich aus, skaliert **zurück auf Originalgröße** → Zoom/Ausschnitt |
| `RandomHorizontalFlip` | **Checkbox** (p = 0,5) | spiegelt links↔rechts |
| `RandomVerticalFlip` | **Checkbox** (p = 0,5) | spiegelt oben↔unten |
| `RandomRotation` | **Stärke** (max. Grad) | dreht um zufälligen Winkel ±Grad |
| `RandomPerspective` | **Stärke** (Verzerrung) | „kippt" das Bild perspektivisch (p = 0,5) |
| `RandomAdjustSharpness` | **Stärke** (Faktor; 1 = neutral) | <1 weichzeichnen, >1 schärfen |
| `RandomAutocontrast` | **Wahrscheinlichkeit** | streckt den Kontrast auf vollen Bereich |

> ⚠️ **Auflösung bleibt erhalten.** `RandomResizedCrop` bekommt als Zielgröße die Originalgröße –
> egal wie stark gecroppt wird, am Ende ist das Bild wieder 24×24 (bzw. deine Größe). Dadurch bleibt
> die **Feature-Zahl im CSV konstant**.

> 🔄 **Reihenfolge zählt!** Jede Transformation hat eine **Reihenfolge-Nummer**. Das `Compose`-Objekt
> wird genau in dieser Reihenfolge gebaut. Bei einem Kreis sieht man deutlich: *erst drehen, dann kippen*
> ≠ *erst kippen, dann drehen*. (Eigenes Experiment am Ende.)


In [ ]:
# --- Metadaten aller Transformationen (steuert spaeter automatisch die UI) -----------
# typ: "scale"  -> Slider fuer Staerke
#      "flip"   -> Checkbox (an => p=0.5)
#      "prob"   -> Slider fuer Wahrscheinlichkeit
TRANSFORM_DEFS = [
    dict(name="RandomResizedCrop",    typ="scale", label="ResizedCrop · Skala (1 = aus)",
         lo=0.2, hi=1.0, step=0.05, default=1.0, neutral=1.0,
         hilfe="1.0 = kein Crop, kleiner = stärkerer Zoom-Ausschnitt"),
    dict(name="RandomHorizontalFlip", typ="flip",  label="↔ H-Flip (p = 0,5)",
         default=False, hilfe="an => 50% der Bilder horizontal gespiegelt"),
    dict(name="RandomVerticalFlip",   typ="flip",  label="↕ V-Flip (p = 0,5)",
         default=False, hilfe="an => 50% der Bilder vertikal gespiegelt"),
    dict(name="RandomRotation",       typ="scale", label="Rotation · max ° (0 = aus)",
         lo=0.0, hi=180.0, step=5.0, default=0.0, neutral=0.0,
         hilfe="dreht zufällig um ±Winkel; 0 = aus"),
    dict(name="RandomPerspective",    typ="scale", label="Perspektive · Verzerrung (0 = aus)",
         lo=0.0, hi=1.0, step=0.05, default=0.0, neutral=0.0,
         hilfe="0 = aus; größer = stärkere Verkippung (p = 0,5)"),
    dict(name="RandomAdjustSharpness",typ="scale", label="Sharpness · Faktor (1 = neutral)",
         lo=0.0, hi=4.0, step=0.25, default=1.0, neutral=1.0,
         hilfe="<1 weichzeichnen, 1 unverändert, >1 schärfen"),
    dict(name="RandomAutocontrast",   typ="prob",  label="Autocontrast · p (0 = aus)",
         lo=0.0, hi=1.0, step=0.1, default=0.0, neutral=0.0,
         hilfe="Wahrscheinlichkeit, den Kontrast voll auszureizen"),
]
NAMES = [d["name"] for d in TRANSFORM_DEFS]


def baue_compose(spec, size):
    """spec = Liste von dicts {name, enabled, order, value}. size = (H, W).
    Baut ein torchvision-Compose in der per 'order' gewuenschten Reihenfolge.
    Arbeitet komplett auf PIL 'L' -> Ausgabe bleibt uint8 und gleich gross."""
    H, W = size
    aktive = [t for t in spec if t["enabled"]]
    aktive = sorted(aktive, key=lambda t: t["order"])
    tfs = []
    for t in aktive:
        n, v = t["name"], t["value"]
        if n == "RandomResizedCrop":
            tfs.append(transforms.RandomResizedCrop(size=(H, W), scale=(float(v), 1.0), antialias=True))
        elif n == "RandomHorizontalFlip":
            tfs.append(transforms.RandomHorizontalFlip(p=0.5 if v else 0.0))
        elif n == "RandomVerticalFlip":
            tfs.append(transforms.RandomVerticalFlip(p=0.5 if v else 0.0))
        elif n == "RandomRotation":
            tfs.append(transforms.RandomRotation(degrees=float(v)))
        elif n == "RandomPerspective":
            tfs.append(transforms.RandomPerspective(distortion_scale=float(v), p=0.5))
        elif n == "RandomAdjustSharpness":
            tfs.append(transforms.RandomAdjustSharpness(sharpness_factor=float(v), p=1.0))
        elif n == "RandomAutocontrast":
            tfs.append(transforms.RandomAutocontrast(p=float(v)))
    return transforms.Compose(tfs)


print("Definierte Transformationen:", NAMES)

## 4 · Augmentieren, CSV, PNG & JSON

Hier sind die „Arbeitspferde":
- **`augmentiere`** – wendet das Compose auf ein PIL-Bild an, liefert ein `uint8`-NumPy-Array (gleiche Größe).
- **`erzeuge_datensatz`** – pro Klasse `n` augmentierte Varianten → Zeilen `feature1…featureN, class`,
  dann **mit Seed gemischt**.
- **`speichere_csv` / `speichere_pngs`** – Export.
- **`zustand_dict` / `lade_zustand`** – die **JSON**, die *Label-Encoding + alle Parameter + Reihenfolge*
  enthält. Wir speichern die Transformationen als **Daten** (nicht als pickle-Objekt) – dadurch ist die
  JSON lesbar, sicher und exakt reproduzierbar wiederherstellbar.


In [ ]:
def augmentiere(bild_pil, compose):
    """PIL 'L' -> augmentiertes uint8-Array (H, W), Werte 0..255."""
    aug = compose(bild_pil)
    return np.array(aug, dtype=np.uint8)


def erzeuge_datensatz(items, encoding, size, spec, n_pro_klasse, seed):
    """Gibt (DataFrame, augmentierte_bilder) zurueck.
    augmentierte_bilder = Liste (klasse, index, uint8-Array) fuer optionalen PNG-Export."""
    H, W = size
    torch.manual_seed(seed); np.random.seed(seed)
    compose = baue_compose(spec, size)

    zeilen, bilder = [], []
    for klasse, bild in items:
        for i in range(n_pro_klasse):
            arr = augmentiere(bild, compose)
            bilder.append((klasse, i, arr))
            zeilen.append(list(arr.flatten().astype(int)) + [encoding[klasse]])

    spalten = [f"feature{i+1}" for i in range(H * W)] + ["class"]
    df = pd.DataFrame(zeilen, columns=spalten)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)   # mischen mit Seed
    return df, bilder


def speichere_csv(df, pfad):
    df.to_csv(pfad, index=False)
    return pfad


def speichere_pngs(bilder, ordner):
    """Dateiname = urspruengliche Klasse + Nummerierung, Graustufen-PNG."""
    os.makedirs(ordner, exist_ok=True)
    n = 0
    for klasse, idx, arr in bilder:
        Image.fromarray(arr, mode="L").save(os.path.join(ordner, f"{klasse}_{idx:03d}.png"))
        n += 1
    return n


def zustand_dict(encoding, size, spec, n_pro_klasse, seed, save_png):
    """Vollstaendiger, serialisierbarer Zustand -> JSON."""
    H, W = size
    return {
        "version": 1,
        "label_encoding": encoding,
        "image_height": H, "image_width": W, "num_features": H * W,
        "n_pro_klasse": int(n_pro_klasse), "seed": int(seed),
        "save_png": bool(save_png),
        "transforms": spec,        # Liste von {name, enabled, order, value}
    }


def speichere_zustand(state, pfad):
    with open(pfad, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, ensure_ascii=False)
    return pfad


def lade_zustand(pfad):
    with open(pfad, "r", encoding="utf-8") as f:
        return json.load(f)


print("Engine-Funktionen bereit ✔")

## 5 · Die Gradio-App 🎛️

Das Interface ist auf **ein Browserfenster** ausgelegt – drei Zeilen:

1. **Quelle:** Ordner laden *oder* das eingebettete Beispiel nutzen. Beim Start wird das Beispiel
   **automatisch geladen**, du siehst sofort eine Vorschau.
2. **Links die Regler, rechts die Live-Vorschau** (Original + 3 zufällige Augmentierungen, 2×2).
   *Regler auf Neutralwert = Transformation aus.* Über das kleine **Reihenf.**-Feld stellst du die
   Reihenfolge ein (kleiner = früher).
3. **Datensatz erzeugen:** links Bilder pro Klasse + Seed, rechts PNG-Option, Dateinamen und der Button.

Tabelle, Downloads und das Speichern/Laden der **JSON**-Konfiguration liegen darunter im
ausklappbaren Bereich.


In [ ]:
# ---------------------------------------------------------------- App-Status (global)
STATE = {"items": None, "encoding": None, "size": None,
         "orders": list(range(1, len(TRANSFORM_DEFS) + 1))}   # aktuelle Reihenfolge-Permutation

VORSCHAU_N = 3   # Original + 3 Augmentierungen -> kompaktes 2x2-Raster


def _aktiv(d, v):
    """Ob eine Transformation wirkt: Checkbox an bzw. Regler abweichend vom Neutralwert."""
    if d["typ"] == "flip":
        return bool(v)
    return float(v) != float(d["neutral"])


def _spec_aus_ui(order_list, value_list):
    """Baut die spec-Liste; 'enabled' wird aus dem Wert abgeleitet (Neutral = aus)."""
    spec = []
    for d, od, va in zip(TRANSFORM_DEFS, order_list, value_list):
        spec.append(dict(name=d["name"], enabled=_aktiv(d, va), order=int(od), value=va))
    return spec


def aktion_reihenfolge(*orders):
    """Hält die Reihenfolge-Felder als saubere Permutation 1..n.
    Ändert man ein Feld auf einen belegten Wert, *tauschen* die beiden Felder."""
    orders = [int(o) for o in orders]
    n = len(orders)
    prev = STATE.get("orders", list(range(1, n + 1)))

    geaendert = [i for i in range(n) if orders[i] != prev[i]]
    if len(geaendert) == 1:                      # genau ein Feld geändert -> sauberer Tausch
        i = geaendert[0]
        ziel, alt = orders[i], prev[i]
        for j in range(n):
            if j != i and orders[j] == ziel:     # wer auf 'ziel' saß, bekommt den alten Wert
                orders[j] = alt

    if sorted(orders) != list(range(1, n + 1)):  # Sicherheitsnetz: auf Permutation normalisieren
        rang = sorted(range(n), key=lambda k: (orders[k], k))
        fixed = [0] * n
        for r, k in enumerate(rang, start=1):
            fixed[k] = r
        orders = fixed

    STATE["orders"] = orders
    return [gr.update(value=o) for o in orders]


def aktion_laden(ordner):
    try:
        items, enc, size = lade_ordner(ordner)
        STATE.update(items=items, encoding=enc, size=size)
        H, W = size
        info = f"✅ {len(items)} Bilder · {H}×{W} ({H*W} Features) · Encoding: {enc}"
        klassen = [k for k, _ in items]
        return info, gr.update(choices=klassen, value=klassen[0])
    except Exception as e:
        return f"❌ Fehler: {e}", gr.update(choices=[], value=None)


def _fig_vorschau(klasse, spec, seed):
    """Matplotlib-Figur: Original + VORSCHAU_N Augmentierungen als 2x2-Raster."""
    items, size = STATE["items"], STATE["size"]
    bild = dict(items)[klasse]
    torch.manual_seed(seed); np.random.seed(seed)
    compose = baue_compose(spec, size)

    fig, axes = plt.subplots(2, 2, figsize=(4.6, 4.9))
    axes = axes.ravel()
    axes[0].imshow(np.array(bild), cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Original", fontsize=9); axes[0].axis("off")
    for j in range(VORSCHAU_N):
        arr = augmentiere(bild, compose)
        axes[j + 1].imshow(arr, cmap="gray", vmin=0, vmax=255)
        axes[j + 1].set_title(f"Aug {j+1}", fontsize=9); axes[j + 1].axis("off")
    reihenfolge = " → ".join(t["name"].replace("Random", "")
                             for t in sorted([s for s in spec if s["enabled"]],
                                             key=lambda x: x["order"])) or "—"
    fig.suptitle(f"{klasse}   ·   {reihenfolge}", fontsize=9)
    fig.tight_layout()
    return fig


def aktion_vorschau(klasse, seed, *ui):
    if STATE["items"] is None:
        fig, ax = plt.subplots(figsize=(4.6, 2)); ax.axis("off")
        ax.text(0.5, 0.5, "Bitte Beispiel/Ordner laden.", ha="center", va="center")
        return fig
    if klasse is None:
        klasse = STATE["items"][0][0]
    n = len(TRANSFORM_DEFS)
    spec = _spec_aus_ui(ui[0:n], ui[n:2*n])
    return _fig_vorschau(klasse, spec, int(seed))


def aktion_erzeugen(n_pro_klasse, seed, save_png, out_csv, out_png_dir, *ui):
    if STATE["items"] is None:
        return "❌ Bitte zuerst Beispiel/Ordner laden.", None, None
    n = len(TRANSFORM_DEFS)
    spec = _spec_aus_ui(ui[0:n], ui[n:2*n])
    df, bilder = erzeuge_datensatz(STATE["items"], STATE["encoding"], STATE["size"],
                                   spec, int(n_pro_klasse), int(seed))
    speichere_csv(df, out_csv)

    state = zustand_dict(STATE["encoding"], STATE["size"], spec, n_pro_klasse, seed, save_png)
    json_pfad = os.path.splitext(out_csv)[0] + "_config.json"
    speichere_zustand(state, json_pfad)

    dateien = [out_csv, json_pfad]
    info = (f"✅ {df.shape[0]} Zeilen × {df.shape[1]} Spalten "
            f"({len(STATE['encoding'])} Klassen × {int(n_pro_klasse)}) · "
            f"CSV: {out_csv} · JSON: {json_pfad}")
    if save_png:
        anz = speichere_pngs(bilder, out_png_dir)
        info += f" · {anz} PNGs in '{out_png_dir}/'"
    return info, df.head(6), dateien


def aktion_config_speichern(n_pro_klasse, seed, save_png, *ui):
    if STATE["encoding"] is None:
        return "❌ Bitte zuerst Beispiel/Ordner laden.", None
    n = len(TRANSFORM_DEFS)
    spec = _spec_aus_ui(ui[0:n], ui[n:2*n])
    state = zustand_dict(STATE["encoding"], STATE["size"], spec, n_pro_klasse, seed, save_png)
    pfad = "konfiguration.json"
    speichere_zustand(state, pfad)
    return f"✅ gespeichert: {pfad}", pfad


def aktion_config_laden(datei):
    """Liest JSON und stellt ALLE Bedienelemente wieder her."""
    if datei is None:
        return ["⚠️ Keine Datei."] + [gr.update()] * (2 * len(TRANSFORM_DEFS) + 3)
    state = lade_zustand(datei if isinstance(datei, str) else datei.name)
    spec_map = {t["name"]: t for t in state["transforms"]}

    order_upd, value_upd, orders_plain = [], [], []
    for d in TRANSFORM_DEFS:
        t = spec_map.get(d["name"], dict(order=1, value=d["default"]))
        order_upd.append(gr.update(value=int(t["order"])))
        value_upd.append(gr.update(value=t["value"]))
        orders_plain.append(int(t["order"]))
    STATE["orders"] = orders_plain   # getrackte Reihenfolge mit geladener Konfig synchronisieren

    info = (f"✅ geladen · Encoding: {state.get('label_encoding')} · "
            f"{state.get('image_height')}×{state.get('image_width')}")
    extra = [gr.update(value=state.get("n_pro_klasse", 20)),
             gr.update(value=state.get("seed", 42)),
             gr.update(value=state.get("save_png", False))]
    return [info] + order_upd + value_upd + extra

In [ ]:
CSS = """
.gradio-container {max-width: 1150px !important; margin: auto;}
#tf-row {gap: 4px !important; margin: 0 !important;}
.compact-md p {margin: 2px 0 !important;}
"""
THEME = gr.themes.Soft(spacing_size="sm", text_size="sm", radius_size="sm")

# Gradio 6 erwartet theme/css in launch(), ältere Versionen (4/5) in Blocks():
_GR_MAJOR = int(gr.__version__.split(".")[0])
_blocks_kw = {} if _GR_MAJOR >= 6 else dict(theme=THEME, css=CSS)
_launch_kw = dict(theme=THEME, css=CSS) if _GR_MAJOR >= 6 else {}

with gr.Blocks(title="Datenaugmentierung – Bilder-Werkstatt", **_blocks_kw) as demo:

    gr.Markdown("### 🧪 Datenaugmentierung – Bilder-Werkstatt", elem_classes="compact-md")

    # ===== Zeile 1: Quelle laden =====================================================
    with gr.Row():
        ordner_tb   = gr.Textbox(value="beispiel_formen", label="📂 Ordner (Dateiname = Klasse)", scale=4)
        laden_btn   = gr.Button("Ordner laden", scale=1)
        beispiel_btn= gr.Button("⭐ Eingebettetes Beispiel", variant="primary", scale=1)
    lade_info = gr.Markdown(elem_classes="compact-md")

    # ===== Zeile 2: links Regler, rechts Live-Vorschau ===============================
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            gr.Markdown("**🎚️ Augmentierungen** — Regler auf Neutral = aus · *Reihenf.* kleiner = früher",
                        elem_classes="compact-md")
            order_comps, value_comps = [], []
            for d in TRANSFORM_DEFS:
                with gr.Row(elem_id="tf-row", equal_height=True):
                    od = gr.Number(value=NAMES.index(d["name"]) + 1, precision=0, label="Reihenf.",
                                   minimum=1, maximum=len(TRANSFORM_DEFS), scale=0, min_width=78)
                    if d["typ"] == "flip":
                        va = gr.Checkbox(value=d["default"], label=d["label"], scale=3)
                    else:
                        va = gr.Slider(minimum=d["lo"], maximum=d["hi"], step=d["step"],
                                       value=d["default"], label=d["label"], scale=3)
                order_comps.append(od); value_comps.append(va)
        with gr.Column(scale=1):
            vorschau_plot = gr.Plot(label="👁️ Live-Vorschau")
            with gr.Row():
                klasse_dd     = gr.Dropdown(choices=[], label="Klasse", scale=2)
                vorschau_seed = gr.Number(value=0, precision=0, label="Seed", scale=1, min_width=80)
                wuerfel_btn   = gr.Button("🎲 würfeln", scale=0, min_width=90)

    # ===== Zeile 3: erzeugen & speichern =============================================
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            n_slider = gr.Slider(1, 100, value=20, step=1, label="Bilder pro Klasse (max. 100)")
            seed_num = gr.Number(value=42, precision=0, label="Random-Seed", min_width=120)
        with gr.Column(scale=1):
            with gr.Row():
                png_check  = gr.Checkbox(value=False, label="PNGs speichern", scale=1, min_width=120)
                out_csv_tb = gr.Textbox(value="augmented_dataset.csv", label="CSV-Datei", scale=2)
                out_png_tb = gr.Textbox(value="augmented_png", label="PNG-Ordner", scale=2)
            erzeugen_btn = gr.Button("📦 Datensatz erzeugen", variant="primary")
    erzeugen_info = gr.Markdown(elem_classes="compact-md")

    # ===== Ausklappbar: Tabelle, Downloads, JSON-Konfiguration =======================
    with gr.Accordion("📄 Tabelle · Downloads · Konfiguration (JSON) speichern/laden", open=False):
        vorschau_df    = gr.Dataframe(label="erste Zeilen (feature… , class)", interactive=False)
        download_files = gr.File(label="erzeugte Dateien", file_count="multiple")
        with gr.Row():
            cfg_save_btn = gr.Button("💾 Konfiguration speichern", scale=1)
            cfg_file_out = gr.File(label="Konfig-Download", scale=1)
            cfg_upload   = gr.File(label="Konfiguration laden (.json)", file_types=[".json"], scale=1)
        cfg_info = gr.Markdown(elem_classes="compact-md")

    # ===== Verkabelung ===============================================================
    UI_INPUTS = order_comps + value_comps                 # Reihenfolge fix: erst order, dann value
    vorschau_inputs = [klasse_dd, vorschau_seed] + UI_INPUTS

    laden_btn.click(aktion_laden, [ordner_tb], [lade_info, klasse_dd]).then(
        aktion_vorschau, vorschau_inputs, [vorschau_plot])
    beispiel_btn.click(lambda: "beispiel_formen", None, [ordner_tb]).then(
        aktion_laden, [ordner_tb], [lade_info, klasse_dd]).then(
        aktion_vorschau, vorschau_inputs, [vorschau_plot])

    for comp in value_comps + [klasse_dd, vorschau_seed]:
        comp.change(aktion_vorschau, vorschau_inputs, [vorschau_plot])
    # Reihenfolge-Felder: erst Permutation sauber halten (tauschen), dann Vorschau aktualisieren
    for comp in order_comps:
        comp.change(aktion_reihenfolge, order_comps, order_comps).then(
            aktion_vorschau, vorschau_inputs, [vorschau_plot])
    wuerfel_btn.click(lambda s: int(s) + 1, [vorschau_seed], [vorschau_seed])

    erzeugen_btn.click(aktion_erzeugen,
                       [n_slider, seed_num, png_check, out_csv_tb, out_png_tb] + UI_INPUTS,
                       [erzeugen_info, vorschau_df, download_files])
    cfg_save_btn.click(aktion_config_speichern,
                       [n_slider, seed_num, png_check] + UI_INPUTS,
                       [cfg_info, cfg_file_out])
    cfg_upload.change(aktion_config_laden, [cfg_upload],
                      [cfg_info] + order_comps + value_comps + [n_slider, seed_num, png_check])

    # Beim Start: eingebettetes Beispiel automatisch laden + Vorschau zeigen
    demo.load(aktion_laden, [ordner_tb], [lade_info, klasse_dd]).then(
        aktion_vorschau, vorschau_inputs, [vorschau_plot])

# Starten (im Notebook inline)
demo.launch(**_launch_kw, inbrowser = True)

## 6 · 🔬 Mini-Experiment: Reihenfolge zählt!

Die schönste Einsicht zum Schluss – **ganz ohne Gradio**, direkt im Code, damit man es sieht.
Wir nehmen den **Kreis** und vergleichen zwei Pipelines mit *denselben* Transformationen, nur in
**unterschiedlicher Reihenfolge**:

- **A:** erst `RandomRotation`, dann `RandomPerspective`
- **B:** erst `RandomPerspective`, dann `RandomRotation`

Bei gleichem Seed sind Einzeloperationen identisch – aber das **Ergebnis** unterscheidet sich,
weil die zweite Operation auf einem anderen Zwischenbild arbeitet. **Transformationen kommutieren nicht.**


In [ ]:
import matplotlib.pyplot as plt
items, enc, size = lade_ordner(BEISPIEL_ORDNER)
kreis = dict(items)["kreis"]

spec_A = [
    dict(name="RandomRotation",    enabled=True, order=1, value=45),
    dict(name="RandomPerspective", enabled=True, order=2, value=0.6),
]
spec_B = [
    dict(name="RandomPerspective", enabled=True, order=1, value=0.6),
    dict(name="RandomRotation",    enabled=True, order=2, value=45),
]

def zeige(spec, titel, ax_row, axes):
    torch.manual_seed(123); np.random.seed(123)        # gleicher Seed fuer beide!
    compose = baue_compose(spec, size)
    for j in range(4):
        arr = augmentiere(kreis, compose)
        axes[ax_row][j].imshow(arr, cmap="gray", vmin=0, vmax=255)
        axes[ax_row][j].axis("off")
    axes[ax_row][0].set_ylabel(titel, rotation=0, ha="right", va="center", fontsize=10)

fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
zeige(spec_A, "A: Rotation→Perspektive", 0, axes)
zeige(spec_B, "B: Perspektive→Rotation", 1, axes)
fig.suptitle("Gleiche Transformationen, gleicher Seed – andere Reihenfolge, anderes Ergebnis", fontsize=11)
fig.tight_layout()
plt.show()

### 🧭 Reflexionsfragen für die Studierenden
1. Warum bleibt das **Label** bei allen Augmentierungen gleich – und wo wäre das *nicht* mehr so?
   (Tipp: Was passiert mit einem `6`-Ziffernbild bei `RandomVerticalFlip`?)
2. Warum brauchen wir den **Seed**? Was bedeutet *Reproduzierbarkeit* für ein Experiment?
3. `RandomResizedCrop` skaliert zurück auf die Originalgröße. Welche **Information geht verloren**,
   und warum ist das für Augmentierung trotzdem ok (oder sogar erwünscht)?
4. Erkläre an deinem eigenen Beispiel, warum `A ≠ B` im Experiment oben gilt.
5. Die Werte landen als **Integer 0–255** im CSV. Welche **Normalisierung** würde die CNN-App
   vermutlich noch davorschalten – und warum?


---
## 📎 Hinweise für die Lehrkraft

**torchvision vs. „plain"** – bewusst torchvision als Engine, weil das Lernziel das *Prinzip* ist,
nicht das Implementieren von Interpolation/Resampling. Der „plain"-Weg (Augmentierung von Hand in
NumPy: Rotationsmatrix, bilineare Interpolation, …) eignet sich als **optionaler Vertiefungs-Exkurs**
für eine spätere Sitzung, in der es um *Bildgeometrie* geht – nicht als Basis dieser Stunde.

**Didaktische Sollbruchstellen / Erweiterungen**
- Eigene Formen in Photoshop malen lassen (Graustufe, ≤28×28) → sofort einsetzbar.
- Train/Test-Split: Augmentierung **nur auf Trainingsdaten** – guter Anlass, Datenleckage zu besprechen.
- „Kaputte" Augmentierung als Fehlerforensik: z. B. `RandomVerticalFlip` bei Ziffern → Label nicht mehr gültig.
- Die erzeugte CSV direkt in die nachgelagerte CNN-App geben und Accuracy *mit/ohne* Augmentierung vergleichen.

**Technik**
- Alle Transformationen laufen auf PIL `'L'` → Ausgabe bleibt `uint8`, Größe konstant → Feature-Zahl stabil.
- Die JSON speichert die Transformationen als **Daten** (kein pickle): lesbar, sicher, exakt rekonstruierbar.
- Leerflächen (durch Rotation/Perspektive) werden mit 0 (schwarz) gefüllt – passt zu hellen Formen auf dunklem Grund.
